# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Logistic Regression, then Random Forest. Per the skill's own table, this is a "yes/no with an observed label" question (is_declining), so the recommended path is readable-first: Logistic Regression as the interpretable baseline model, Random Forest only if it earns its added complexity over LR. Evaluated at Precision@K (10/20/50) to match the ranking nature of the actual decision ("which pages to review first"), not raw accuracy.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [1]:
import duckdb, getpass
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

momentum_query = f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_prev
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT mar.*, feb.imp_prev
    FROM mar LEFT JOIN feb
      ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
    WHERE mar.imp_last IS NOT NULL AND feb.imp_prev >= 100
"""
df = conn.execute(momentum_query).df()

content_query = f"""
    SELECT content_hash_id, content_type, word_count, content_created_date
    FROM read_parquet('{REL}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
"""
content_df = conn.execute(content_query).df()
df = df.merge(content_df, on="content_hash_id", how="inner")  # inner: drop items missing from dim_content, same as w03

df["is_declining"] = (df["imp_last"] < 0.8 * df["imp_prev"]).astype(int)
df["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df["content_created_date"])).dt.days
df["word_count"] = df["word_count"].fillna(df["word_count"].median())
df["content_type"] = df["content_type"].fillna("unknown")

# Client-grouped split — same principle as notebooks 01/02: a client never appears in both sides
np.random.seed(42)
clients = df["client_hash_id"].unique()
np.random.shuffle(clients)
test_clients = set(clients[:max(1, int(len(clients) * 0.2))])
test_mask = df["client_hash_id"].isin(test_clients)

train_df, test_df = df[~test_mask].copy(), df[test_mask].copy()
print(f"Train: {len(train_df)} rows, {len(clients) - len(test_clients)} clients")
print(f"Test:  {len(test_df)} rows, {len(test_clients)} clients")
print(f"Decline rate — train: {train_df['is_declining'].mean():.3f}, test: {test_df['is_declining'].mean():.3f}")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train: 53715 rows, 28 clients
Test:  23023 rows, 6 clients
Decline rate — train: 0.188, test: 0.165


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Because only ~34 clients exist total, a single 80/20 client-holdout split is unstable — Precision@10 for Random Forest ranged from 0.40 to 1.00 across 5 different random seeds, driven entirely by which handful of clients landed in the test set, not by any real difference in model quality between runs. The trustworthy comparison is the mean across multiple splits, not any one split's number.

In [2]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Baseline, recomputed on the TEST split only (fair comparison) ---
test_df["baseline_score"] = np.where(
    (test_df["imp_prev"] >= 100) & (test_df["avg_position_mar"] > 10),
    test_df["imp_prev"], 0
)
print("Baseline nonzero on test:", (test_df["baseline_score"] > 0).sum(), "of", len(test_df))

# --- Honest feature set (no label-derived or future-window inputs) ---
content_type_dummies_train = pd.get_dummies(train_df["content_type"], prefix="ct", drop_first=True)
content_type_dummies_test = pd.get_dummies(test_df["content_type"], prefix="ct", drop_first=True)
content_type_dummies_test = content_type_dummies_test.reindex(columns=content_type_dummies_train.columns, fill_value=0)

feature_cols = ["avg_position_mar", "imp_prev", "content_age_days", "word_count"]
X_train = pd.concat([train_df[feature_cols].fillna(0).reset_index(drop=True), content_type_dummies_train.reset_index(drop=True)], axis=1)
X_test = pd.concat([test_df[feature_cols].fillna(0).reset_index(drop=True), content_type_dummies_test.reset_index(drop=True)], axis=1)
y_train, y_test = train_df["is_declining"].values, test_df["is_declining"].values

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(X_train, y_train)
lr_score = lr.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

results = []
for k in (10, 20, 50):
    results.append({
        "k": k,
        "baseline": precision_at_k(test_df["baseline_score"].values, y_test, k),
        "logistic_regression": precision_at_k(lr_score, y_test, k),
        "random_forest": precision_at_k(rf_score, y_test, k),
    })
comparison = pd.DataFrame(results)
comparison["base_rate"] = y_test.mean()
print(comparison.to_string(index=False))

Baseline nonzero on test: 7931 of 23023
 k  baseline  logistic_regression  random_forest  base_rate
10      0.60                 0.40           1.00   0.165226
20      0.45                 0.45           0.95   0.165226
50      0.36                 0.40           0.72   0.165226


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random Forest's apparent large win in the first split (Precision@10 = 1.00 vs. baseline 0.60) did not survive scrutiny: it was driven almost entirely by 1-2 small, easy-to-separate clients (n=785, n=219) while performing only marginally above the base rate on the client making up 79.5% of that test set (Precision@20 = 0.250 vs. 0.186 decline rate). The top permutation-importance feature, word_count, is plausibly acting as a client fingerprint rather than a genuine content-quality signal, given its mean varies substantially across clients (1,101 to 4,668).

In [3]:
# Sanity-check what the model leans on — permutation importance, not just built-in feature_importances_
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print(importance_df.head(8).to_string(index=False))

# 3 concrete wrong cases from the top-20 ranked by the RF model
test_df["rf_score"] = rf_score
wrong_top20 = test_df.sort_values("rf_score", ascending=False).head(20)
wrong_top20 = wrong_top20[wrong_top20["is_declining"] == 0]  # model flagged it, but it wasn't actually declining
print(wrong_top20[["content_hash_id", "avg_position_mar", "imp_prev", "content_age_days", "rf_score"]].head(3).to_string(index=False))

           feature  importance
        word_count    0.072466
  avg_position_mar    0.039157
  content_age_days    0.011053
 ct_feedly article    0.005193
          imp_prev    0.002107
ct_keyword article    0.000437
         content_hash_id  avg_position_mar  imp_prev  content_age_days  rf_score
content_0fc553ca66178e8c         10.551714     123.0               254  0.755772


In [4]:
# Is word_count basically a client fingerprint?
print(df.groupby("client_hash_id")["word_count"].agg(["mean", "std", "count"]).round(1))

# Does RF's performance hold across MULTIPLE test clients, or is it 1-2 clients carrying the whole result?
test_df["rf_score"] = rf_score
for client in test_df["client_hash_id"].unique():
    sub = test_df[test_df["client_hash_id"] == client]
    if len(sub) >= 20:
        p20 = precision_at_k(sub["rf_score"].values, sub["is_declining"].values, min(20, len(sub)))
        print(f"{client}: n={len(sub)}, decline_rate={sub['is_declining'].mean():.3f}, RF Precision@20={p20:.3f}")

                           mean     std  count
client_hash_id                                
client_0797ff3a1fc9a6a5  3475.1   390.5     13
client_08a6a72ff48e62c0  2692.5   259.7   4672
client_08d2847f24cf89c1  1543.2   150.6      9
client_0e1acc6cd57b0eba  1508.5     0.7      2
client_0fa64a184f18a4a0  2758.6   212.6    158
client_157ffe4d4a595515  2899.1   617.7    338
client_1a730cb2640a1abf  2548.8   223.9    286
client_20259bd6705d81d4  3858.1   619.2   2570
client_2094c6eb080311d5  2729.8   532.8    595
client_23a62021009f63c4  4668.4  1433.1  10125
client_3197e6291363b4db  2550.1  1017.7    807
client_3f0ce4d44fe94f3d  2597.7   436.9    973
client_3ffa76342f366962  1155.4   702.7    112
client_400c21c81c8b46ef  1696.5   514.1    785
client_62f4a7e64f5e0096  2716.6   286.0  15379
client_65de48885f4ef01b  2049.7   755.7    570
client_73cda7b4e4f265ea  2734.3   238.3  18318
client_795153d5b7850ccf  1402.7   172.0     70
client_8ae2bfb5aa1ffa1e  1375.3   107.8     10
client_8dbf3a

In [5]:
# Does RF's apparent edge survive across DIFFERENT random client splits, or is it luck-of-the-split?
for seed in (1, 2, 3, 42, 99):
    np.random.seed(seed)
    clients_s = df["client_hash_id"].unique()
    np.random.shuffle(clients_s)
    test_clients_s = set(clients_s[:max(1, int(len(clients_s) * 0.2))])
    test_mask_s = df["client_hash_id"].isin(test_clients_s)

    tr, te = df[~test_mask_s], df[test_mask_s]
    ctd_tr = pd.get_dummies(tr["content_type"], prefix="ct", drop_first=True)
    ctd_te = pd.get_dummies(te["content_type"], prefix="ct", drop_first=True).reindex(columns=ctd_tr.columns, fill_value=0)
    Xtr = pd.concat([tr[feature_cols].fillna(0).reset_index(drop=True), ctd_tr.reset_index(drop=True)], axis=1)
    Xte = pd.concat([te[feature_cols].fillna(0).reset_index(drop=True), ctd_te.reset_index(drop=True)], axis=1)

    rf_s = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(Xtr, tr["is_declining"])
    p10 = precision_at_k(rf_s.predict_proba(Xte)[:, 1], te["is_declining"].values, 10)
    print(f"seed={seed}: test_clients={len(test_clients_s)}, test_n={len(te)}, RF Precision@10={p10:.3f}")

seed=1: test_clients=6, test_n=6560, RF Precision@10=1.000
seed=2: test_clients=6, test_n=3379, RF Precision@10=1.000
seed=3: test_clients=6, test_n=16735, RF Precision@10=0.400
seed=42: test_clients=6, test_n=23023, RF Precision@10=1.000
seed=99: test_clients=6, test_n=5128, RF Precision@10=0.900


In [6]:
def evaluate_seed(seed, feature_cols):
    np.random.seed(seed)
    clients_s = df["client_hash_id"].unique()
    np.random.shuffle(clients_s)
    test_clients_s = set(clients_s[:max(1, int(len(clients_s) * 0.2))])
    test_mask_s = df["client_hash_id"].isin(test_clients_s)
    tr, te = df[~test_mask_s], df[test_mask_s]

    ctd_tr = pd.get_dummies(tr["content_type"], prefix="ct", drop_first=True)
    ctd_te = pd.get_dummies(te["content_type"], prefix="ct", drop_first=True).reindex(columns=ctd_tr.columns, fill_value=0)
    Xtr = pd.concat([tr[feature_cols].fillna(0).reset_index(drop=True), ctd_tr.reset_index(drop=True)], axis=1)
    Xte = pd.concat([te[feature_cols].fillna(0).reset_index(drop=True), ctd_te.reset_index(drop=True)], axis=1)
    ytr, yte = tr["is_declining"].values, te["is_declining"].values

    baseline_score = np.where((te["imp_prev"] >= 100) & (te["avg_position_mar"] > 10), te["imp_prev"], 0)
    lr_s = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(Xtr, ytr)
    rf_s = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(Xtr, ytr)

    return {
        "seed": seed, "test_n": len(te),
        "baseline_p10": precision_at_k(baseline_score, yte, 10),
        "lr_p10": precision_at_k(lr_s.predict_proba(Xte)[:, 1], yte, 10),
        "rf_p10": precision_at_k(rf_s.predict_proba(Xte)[:, 1], yte, 10),
    }

runs = pd.DataFrame([evaluate_seed(s, feature_cols) for s in (1, 2, 3, 42, 99)])
print(runs.to_string(index=False))
print("\nMean ± std across splits:")
print(runs[["baseline_p10", "lr_p10", "rf_p10"]].agg(["mean", "std"]).round(3))

 seed  test_n  baseline_p10  lr_p10  rf_p10
    1    6560           0.6     0.1     1.0
    2    3379           0.3     0.0     1.0
    3   16735           0.0     0.3     0.4
   42   23023           0.6     0.4     1.0
   99    5128           0.2     0.1     0.9

Mean ± std across splits:
      baseline_p10  lr_p10  rf_p10
mean         0.340   0.180   0.860
std          0.261   0.164   0.261


In [7]:
from sklearn.preprocessing import StandardScaler

def evaluate_seed_scaled(seed, feature_cols):
    np.random.seed(seed)
    clients_s = df["client_hash_id"].unique()
    np.random.shuffle(clients_s)
    test_clients_s = set(clients_s[:max(1, int(len(clients_s) * 0.2))])
    test_mask_s = df["client_hash_id"].isin(test_clients_s)
    tr, te = df[~test_mask_s], df[test_mask_s]

    ctd_tr = pd.get_dummies(tr["content_type"], prefix="ct", drop_first=True)
    ctd_te = pd.get_dummies(te["content_type"], prefix="ct", drop_first=True).reindex(columns=ctd_tr.columns, fill_value=0)
    Xtr_raw = pd.concat([tr[feature_cols].fillna(0).reset_index(drop=True), ctd_tr.reset_index(drop=True)], axis=1)
    Xte_raw = pd.concat([te[feature_cols].fillna(0).reset_index(drop=True), ctd_te.reset_index(drop=True)], axis=1)

    scaler = StandardScaler().fit(Xtr_raw)
    Xtr, Xte = scaler.transform(Xtr_raw), scaler.transform(Xte_raw)
    ytr, yte = tr["is_declining"].values, te["is_declining"].values

    baseline_score = np.where((te["imp_prev"] >= 100) & (te["avg_position_mar"] > 10), te["imp_prev"], 0)
    lr_s = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(Xtr, ytr)

    return {"seed": seed, "baseline_p10": precision_at_k(baseline_score, yte, 10),
            "lr_scaled_p10": precision_at_k(lr_s.predict_proba(Xte)[:, 1], yte, 10)}

runs_scaled = pd.DataFrame([evaluate_seed_scaled(s, feature_cols) for s in (1, 2, 3, 42, 99)])
print(runs_scaled.to_string(index=False))

 seed  baseline_p10  lr_scaled_p10
    1           0.6            0.1
    2           0.3            0.0
    3           0.0            0.3
   42           0.6            0.4
   99           0.2            0.1


In [8]:
# Using your seed=42 split (Xtrain/Xtest, lr from earlier)
lr_probs = lr.predict_proba(X_test)[:, 1]
print("Distinct LR probabilities:", len(np.unique(lr_probs.round(4))))
print("Top 10 LR probabilities:", np.sort(lr_probs)[-10:])
print("Overall AUC comparison:")
from sklearn.metrics import roc_auc_score
print("LR AUC:", roc_auc_score(y_test, lr_probs))
print("RF AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

Distinct LR probabilities: 2135
Top 10 LR probabilities: [0.85401563 0.85475352 0.85537551 0.85565835 0.85791818 0.85933316
 0.8602637  0.860419   0.86368697 0.86902871]
Overall AUC comparison:
LR AUC: 0.45016719948696937
RF AUC: 0.6056191231304853


In [9]:
lr_unweighted = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
lr_unweighted_probs = lr_unweighted.predict_proba(X_test)[:, 1]
print("Unweighted LR Precision@10:", precision_at_k(lr_unweighted_probs, y_test, 10))

Unweighted LR Precision@10: 0.4


In [10]:
coef_df = pd.DataFrame({"feature": X_train.columns, "coef": lr.coef_[0]}).sort_values("coef")
print(coef_df)

              feature      coef
2    content_age_days -0.000138
1            imp_prev -0.000019
3          word_count  0.000117
0    avg_position_mar  0.000531
5  ct_keyword article  1.159567
4   ct_feedly article  2.687041


In [11]:
X_train_pos_only = X_train[["avg_position_mar"]]
X_test_pos_only = X_test[["avg_position_mar"]]

lr_pos_only = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_pos_only, y_train)
pos_only_auc = roc_auc_score(y_test, lr_pos_only.predict_proba(X_test_pos_only)[:, 1])
print("Position-only LR AUC:", pos_only_auc)
print("Position-only Precision@10:", precision_at_k(lr_pos_only.predict_proba(X_test_pos_only)[:, 1], y_test, 10))

Position-only LR AUC: 0.5412589060761758
Position-only Precision@10: 0.3


In [12]:
print("X_train index matches train_df index:", (X_train.index == train_df.reset_index(drop=True).index).all())
print("Any duplicate content_hash_id in train_df:", train_df["content_hash_id"].duplicated().sum())
print("train_df content_type value counts:\n", train_df["content_type"].value_counts())
print("test_df content_type value counts:\n", test_df["content_type"].value_counts())

X_train index matches train_df index: True
Any duplicate content_hash_id in train_df: 0
train_df content_type value counts:
 content_type
keyword article       52893
feedly article          561
comparison article      261
Name: count, dtype: int64
test_df content_type value counts:
 content_type
keyword article       22821
feedly article          156
comparison article       46
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.